# Pandas: Data Manipulation

## Overview
Pandas provides data structures for tabular data:
- `DataFrame` — 2D labeled data structure
- `Series` — 1D labeled data structure
- Data cleaning, filtering, grouping
- Time series operations

## Why Pandas for Engineering?
- Process experimental data (sensor readings, measurements)
- Analyze CFD time series (forces, moments, heat flux)
- Prepare data for ML models

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Creating DataFrames
print("=== Creating DataFrames ===")

# From a dictionary
data = {
    'time': [0, 1, 2, 3, 4, 5],
    'force_x': [10, 12, 11, 13, 12, 14],
    'force_y': [5, 6, 5, 7, 6, 8],
    'temperature': [25, 26, 27, 28, 29, 30]
}
df = pd.DataFrame(data)

print(df)
print(f"\nShape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Dtypes:\n{df.dtypes}")

## Data Cleaning & Filtering

Handle missing values, filter rows, and transform data.

In [ ]:
# 2. Data Cleaning & Filtering
print("\n=== Data Cleaning & Filtering ===")

# Add some missing values
df.loc[2, 'force_x'] = np.nan
df.loc[4, 'temperature'] = np.nan

print("DataFrame with missing values:")
print(df)

# Handle missing values
df_clean = df.fillna(method='ffill')  # Forward fill
print(f"\nAfter forward fill:\n{df_clean}")

# Filter rows
high_force = df_clean[df_clean['force_x'] > 11]
print(f"\nRows with force_x > 11:\n{high_force}")

# Add a new column
df_clean['force_mag'] = np.sqrt(df_clean['force_x']**2 + df_clean['force_y']**2)
print(f"\nWith force magnitude:\n{df_clean}")

## Grouping & Aggregation

Group data by categories and compute statistics.

In [ ]:
# 3. Grouping & Aggregation
print("\n=== Grouping & Aggregation ===")

# Create a larger dataset
np.random.seed(42)
n = 100
df_large = pd.DataFrame({
    'time': np.linspace(0, 10, n),
    'sensor_id': np.random.choice(['A', 'B', 'C'], n),
    'value': np.random.randn(n) * 10 + 50
})

# Group by sensor and compute statistics
grouped = df_large.groupby('sensor_id')['value'].agg(['mean', 'std', 'min', 'max'])
print("Statistics by sensor:")
print(grouped)

# Plot grouped data
plt.figure(figsize=(10, 6))
for sensor in ['A', 'B', 'C']:
    subset = df_large[df_large['sensor_id'] == sensor]
    plt.plot(subset['time'], subset['value'], alpha=0.5, label=f'Sensor {sensor}')

plt.xlabel('Time (s)')
plt.ylabel('Value')
plt.title('Sensor Data by Group')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Time Series Operations

Work with time-indexed data (common in CFD and experimental data).

In [ ]:
# 4. Time Series Operations
print("\n=== Time Series Operations ===")

# Create a time series
dates = pd.date_range(start='2024-01-01', periods=100, freq='H')
ts = pd.Series(np.random.randn(100).cumsum(), index=dates)

# Resample to daily frequency
ts_daily = ts.resample('D').mean()

plt.figure(figsize=(12, 6))
plt.plot(ts.index, ts.values, 'b-', alpha=0.5, label='Hourly')
plt.plot(ts_daily.index, ts_daily.values, 'r-o', markersize=4, label='Daily (resampled)')
plt.xlabel('Date')
plt.ylabel('Value')
plt.title('Time Series with Resampling')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Rolling statistics
ts_rolling_mean = ts.rolling(window=10).mean()
ts_rolling_std = ts.rolling(window=10).std()

plt.figure(figsize=(12, 6))
plt.plot(ts.index, ts.values, 'b-', alpha=0.5, label='Original')
plt.plot(ts_rolling_mean.index, ts_rolling_mean.values, 'r-', label='Rolling Mean (10)')
plt.fill_between(ts_rolling_std.index, 
                 ts_rolling_mean - ts_rolling_std, 
                 ts_rolling_mean + ts_rolling_std, 
                 alpha=0.2, color='red', label='±1 Std')
plt.xlabel('Date')
plt.ylabel('Value')
plt.title('Rolling Statistics')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Engineering Application: CFD Time Series Analysis

Analyze force coefficients from a CFD simulation.

In [ ]:
# 5. Engineering Application: CFD Time Series
print("\n=== CFD Time Series Analysis ===")

# Simulate lift and drag coefficients
np.random.seed(42)
t = np.linspace(0, 10, 1000)
cl = 0.5 + 0.1 * np.sin(2 * np.pi * t) + np.random.randn(1000) * 0.01
cd = 0.02 + 0.005 * np.cos(2 * np.pi * t) + np.random.randn(1000) * 0.001

df_cfd = pd.DataFrame({
    'time': t,
    'cl': cl,
    'cd': cd
})

# Compute statistics
stats = df_cfd[['cl', 'cd']].describe()
print("Force Coefficient Statistics:")
print(stats)

# Plot time series
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(t, cl, 'b-', alpha=0.7)
axes[0].axhline(y=cl.mean(), color='r', linestyle='--', label=f'Mean: {cl.mean():.3f}')
axes[0].set_ylabel('Cl (Lift Coefficient)')
axes[0].set_title('Lift Coefficient Time Series')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, cd, 'r-', alpha=0.7)
axes[1].axhline(y=cd.mean(), color='b', linestyle='--', label=f'Mean: {cd.mean():.4f}')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Cd (Drag Coefficient)')
axes[1].set_title('Drag Coefficient Time Series')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Exercises

1. Load a CSV file with experimental data and clean it (handle missing values, outliers).
2. Group data by test condition and compute mean/std for each group.
3. Resample a high-frequency time series to a lower frequency and plot rolling statistics.